# e6 · Overall comparison to the benchmark (`hist_rw`)

Does each model forecast next month's realised variance better than the historical-volatility benchmark `hist_rw` (the forecast for month t+1 is month t's realised variance)? Models come from `research_config.SELECTED`; each is compared with `BENCHMARK`.

**Tests.** Every Elastic Net variant includes `log_rv_1m`, so it nests `hist_rw` (intercept 0, coefficient 1 on `log_rv_1m`, 0 on the rest). For these, MSE on log RV uses Clark-West and QLIKE uses Diebold-Mariano, which is undersized for nested models, so read its p-values with caution. GARCH, GARCH-MIDAS and XGBoost do not nest `hist_rw`, so both losses use Diebold-Mariano. Clark-West was derived for linear models estimated by OLS, so applying it to Elastic Net is an approximation. As in `e3_vix_effect_en.ipynb`, the loss differential is averaged across ETFs each month and tested with Newey-West (Andrews bandwidth). p-values are one-sided, H1: `a` beats `b`.

**Output** in `forecasting/results/`: `e6_losses.csv`, `e6_tests.csv` (read by `portfolio_construction/03_forecast_vs_portfolio.ipynb`). `a` is the model and `b` is `hist_rw`. `gain_t > 0` favours `a`. `loss_ratio` is `a`'s loss over `b`'s; the Clark-West statistic adjusts for nesting, so it can be significant while the ratio is above 1.

In [9]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import norm

MODELS_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "research_config.py").exists())
sys.path.insert(0, str(MODELS_DIR))
from research_config import SELECTED, BENCHMARK, TICKERS
from utils.portfolio_utils import FORECAST_FILES

FORECAST_DIR = MODELS_DIR / "forecasting" / "forecasts"
RESULTS_DIR = MODELS_DIR / "forecasting" / "results"
RESULTS_DIR.mkdir(exist_ok=True)

# Families whose variants nest hist_rw: each Elastic Net includes log_rv_1m
NESTED_FAMILIES = {"Elastic Net"}

df = pd.concat([pd.read_csv(FORECAST_DIR / FORECAST_FILES[fam], parse_dates=["date"]).assign(family=fam)
                for fam in SELECTED], ignore_index=True)
selected = [m for models in SELECTED.values() for m in models]
available = set(df["model"])
models = [m for m in selected if m in available]
assert BENCHMARK in models, f"{BENCHMARK} has no forecasts"
nested = {m: fam in NESTED_FAMILIES for fam, ms in SELECTED.items() for m in ms}
pairs = [(f"E6.{i}", m, BENCHMARK) for i, m in enumerate([m for m in models if m != BENCHMARK], 1)]
print(f"{len(models)} of {len(selected)} models available; {len(pairs)} comparisons with {BENCHMARK}")

11 of 11 models available; 10 comparisons with hist_rw


In [10]:
def andrews_bandwidth(x):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    den = float(xd[:-1] @ xd[:-1])
    rho = 0.0 if den <= 0 else float(xd[1:] @ xd[:-1]) / den
    rho = min(max(rho, -0.97), 0.97)
    a1 = 4.0 * rho ** 2 / ((1 - rho) ** 2 * (1 + rho) ** 2)
    return 0 if a1 <= 0 else int(min(np.floor(1.1447 * (a1 * n) ** (1 / 3)), n // 4))


def newey_west_lrv(x, lag):
    x = np.asarray(x, float); n = len(x); xd = x - x.mean()
    lag = max(0, min(lag, n - 1)); s = float(xd @ xd) / n
    for l in range(1, lag + 1):
        s += 2.0 * (1.0 - l / (lag + 1.0)) * float(xd[l:] @ xd[:-l]) / n
    return max(s, 1e-16)


def hac_t(d):
    """t-statistic of mean(d) with a Newey-West variance."""
    d = np.asarray(d, float)
    return d.mean() / np.sqrt(newey_west_lrv(d, andrews_bandwidth(d)) / len(d))


def qlike(rv, fc):
    r = rv / fc
    return r - np.log(r) - 1

## Losses

Common sample: the (ETF, month) rows where all models have a forecast. `XGBoost` starts one month earlier (2008-01), which is dropped.

In [11]:
d = df[df["model"].isin(models)].copy()
d = d[d.groupby(["etf", "date"])["model"].transform("nunique") == len(models)]
rv_range = d.groupby(["etf", "date"])["actual_RV"].agg(["min", "max"])
assert np.allclose(rv_range["min"], rv_range["max"]), "realised RV differs across forecast files"
d["MSE_log"] = (d["actual_logRV"] - d["fc_log"]) ** 2
d["MSE_level"] = (d["actual_RV"] - d["fc_level"]) ** 2
d["QLIKE"] = qlike(d["actual_RV"], d["fc_level"])
print(f"{d['date'].min():%Y-%m} to {d['date'].max():%Y-%m}, {len(d)} rows")

LOSSES = ["QLIKE", "MSE_log", "MSE_level"]
per_etf = d.groupby(["model", "etf"])[LOSSES].mean().reset_index()
pooled = d.groupby("model")[LOSSES].mean().reset_index().assign(etf="pooled")
losses = pd.concat([per_etf, pooled], ignore_index=True).rename(columns={"etf": "ETF"})
losses.to_csv(RESULTS_DIR / "e6_losses.csv", index=False)
losses[losses["ETF"] == "pooled"].set_index("model").loc[models, LOSSES].round(4)

2008-02 to 2025-11, 11770 rows


,QLIKE,MSE_log,MSE_level
model,,,
hist_rw,0.406,0.603,0.000
garch,0.264,0.474,0.000
garch_midas_baa10y,0.268,0.483,0.000
en_no_macro,0.327,0.523,0.000
en_no_vix,0.380,0.537,0.000
en_vix_only,0.291,0.477,0.002
en_full_macro,0.325,0.490,0.001
xgb_no_macro,0.410,0.589,0.000
xgb_no_vix,0.442,0.577,0.000


## Tests

In [12]:
def one_sided_p(t):
    return float(1 - norm.cdf(t))


loss_idx = losses.set_index(["model", "ETF"])
rows = []
for pair_id, a, b in pairs:
    A = d[d["model"] == a].set_index(["date", "etf"])
    B = d[d["model"] == b].set_index(["date", "etf"])
    dm_qlike = A["QLIKE"] - B["QLIKE"]       # negative favours a
    dm_mse = A["MSE_log"] - B["MSE_log"]     # negative favours a
    # Clark-West adjusted differential; positive favours a
    cw = B["MSE_log"] - (A["MSE_log"] - (B["fc_log"] - A["fc_log"]) ** 2)

    for etf in ["pooled"] + TICKERS:
        def unit(x):
            return x.groupby("date").mean() if etf == "pooled" else x.xs(etf, level="etf")

        mse_test = ("Clark-West", hac_t(unit(cw))) if nested[a] else ("DM", -hac_t(unit(dm_mse)))
        for loss, (test, t) in [("QLIKE", ("DM", -hac_t(unit(dm_qlike)))), ("MSE_log", mse_test)]:
            rows.append({
                "pair": pair_id, "a": a, "b": b, "ETF": etf, "loss": loss, "test": test,
                "loss_ratio": loss_idx.loc[(a, etf), loss] / loss_idx.loc[(b, etf), loss],
                "gain_t": t, "p_value": one_sided_p(t),
            })

tests = pd.DataFrame(rows)
tests.to_csv(RESULTS_DIR / "e6_tests.csv", index=False)
tests[tests["ETF"] == "pooled"].drop(columns="ETF").round(4).reset_index(drop=True)

,pair,a,b,loss,test,loss_ratio,gain_t,p_value
0,E6.1,garch,hist_rw,QLIKE,DM,0.649,4.103,0.000
1,E6.1,garch,hist_rw,MSE_log,DM,0.785,3.843,0.000
2,E6.2,garch_midas_baa10y,hist_rw,QLIKE,DM,0.660,4.057,0.000
3,E6.2,garch_midas_baa10y,hist_rw,MSE_log,DM,0.800,3.534,0.000
4,E6.3,en_no_macro,hist_rw,QLIKE,DM,0.804,3.198,0.001
5,E6.3,en_no_macro,hist_rw,MSE_log,Clark-West,0.867,6.067,0.000
6,E6.4,en_no_vix,hist_rw,QLIKE,DM,0.935,0.523,0.300
7,E6.4,en_no_vix,hist_rw,MSE_log,Clark-West,0.891,5.543,0.000
8,E6.5,en_vix_only,hist_rw,QLIKE,DM,0.717,3.653,0.000
9,E6.5,en_vix_only,hist_rw,MSE_log,Clark-West,0.791,8.012,0.000


## By sector (exploratory)

Each cell is `loss_ratio (p-value)`; a ratio below 1 means the model is better than `hist_rw`. Five sectors times ten models with no multiple-testing correction, so a single sector's p-value is weak evidence; the pooled column is the formal result. Log-RV MSE uses Clark-West for Elastic Net and Diebold-Mariano for the others; QLIKE uses Diebold-Mariano (supporting only for Elastic Net).

In [13]:
SECTORS = {"XLB": "Materials", "XLE": "Energy", "XLF": "Financials", "XLI": "Industrials", "XLY": "Cons. Disc."}


def fmt_p(p):
    return "<0.001" if p < 0.001 else f"{p:.3f}"


def by_sector(loss):
    t = tests[tests["loss"] == loss].copy()
    t["cell"] = [f"{r:.2f} ({fmt_p(p)})" for r, p in zip(t["loss_ratio"], t["p_value"])]
    t["row"] = t["pair"] + ": " + t["a"] + " vs " + t["b"] + " (" + t["test"] + ")"
    table = t.pivot(index="row", columns="ETF", values="cell")[TICKERS + ["pooled"]]
    table = table.loc[t["row"].drop_duplicates()]
    table.columns = [f"{c} {SECTORS[c]}" for c in TICKERS] + ["Pooled"]
    return table.rename_axis(None)


for loss, test in [("MSE_log", "Clark-West (Elastic Net) or DM"), ("QLIKE", "DM")]:
    print(f"{loss}: ratio (p-value), {test}")
    display(by_sector(loss))

MSE_log: ratio (p-value), Clark-West (Elastic Net) or DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw (DM),0.80 (0.001),0.83 (<0.001),0.76 (<0.001),0.73 (<0.001),0.83 (0.005),0.79 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw (DM),0.82 (0.003),0.85 (0.006),0.77 (<0.001),0.75 (<0.001),0.84 (0.008),0.80 (<0.001)
E6.3: en_no_macro vs hist_rw (Clark-West),0.85 (<0.001),0.88 (<0.001),0.88 (<0.001),0.84 (<0.001),0.88 (<0.001),0.87 (<0.001)
E6.4: en_no_vix vs hist_rw (Clark-West),0.89 (<0.001),0.94 (<0.001),0.89 (<0.001),0.84 (<0.001),0.91 (<0.001),0.89 (<0.001)
E6.5: en_vix_only vs hist_rw (Clark-West),0.75 (<0.001),0.87 (<0.001),0.76 (<0.001),0.72 (<0.001),0.89 (<0.001),0.79 (<0.001)
E6.6: en_full_macro vs hist_rw (Clark-West),0.78 (<0.001),0.92 (<0.001),0.81 (<0.001),0.72 (<0.001),0.85 (<0.001),0.81 (<0.001)
E6.7: xgb_no_macro vs hist_rw (DM),0.98 (0.409),0.99 (0.477),0.94 (0.261),0.89 (0.055),1.11 (0.719),0.98 (0.395)
E6.8: xgb_no_vix vs hist_rw (DM),1.03 (0.602),0.96 (0.379),0.96 (0.394),0.86 (0.059),0.97 (0.370),0.96 (0.335)
E6.9: xgb_vix_only vs hist_rw (DM),0.91 (0.252),0.94 (0.304),0.83 (0.030),0.73 (<0.001),0.91 (0.140),0.86 (0.028)
E6.10: xgb_full_macro vs hist_rw (DM),0.94 (0.341),0.93 (0.290),0.82 (0.029),0.78 (0.011),0.90 (0.149),0.87 (0.069)


QLIKE: ratio (p-value), DM


,XLB Materials,XLE Energy,XLF Financials,XLI Industrials,XLY Cons. Disc.,Pooled
E6.1: garch vs hist_rw (DM),0.67 (<0.001),0.73 (<0.001),0.58 (<0.001),0.62 (<0.001),0.67 (<0.001),0.65 (<0.001)
E6.2: garch_midas_baa10y vs hist_rw (DM),0.66 (<0.001),0.76 (0.003),0.59 (<0.001),0.64 (<0.001),0.69 (<0.001),0.66 (<0.001)
E6.3: en_no_macro vs hist_rw (DM),0.83 (0.009),0.85 (0.025),0.77 (0.003),0.81 (0.002),0.79 (<0.001),0.80 (<0.001)
E6.4: en_no_vix vs hist_rw (DM),0.92 (0.277),1.06 (0.618),0.88 (0.185),0.94 (0.350),0.91 (0.169),0.93 (0.300)
E6.5: en_vix_only vs hist_rw (DM),0.70 (<0.001),0.84 (0.033),0.67 (0.004),0.66 (<0.001),0.75 (0.003),0.72 (<0.001)
E6.6: en_full_macro vs hist_rw (DM),0.78 (0.033),1.03 (0.551),0.75 (0.012),0.71 (<0.001),0.78 (0.003),0.80 (0.022)
E6.7: xgb_no_macro vs hist_rw (DM),1.05 (0.624),1.11 (0.699),0.95 (0.402),0.84 (0.037),1.14 (0.721),1.01 (0.529)
E6.8: xgb_no_vix vs hist_rw (DM),1.18 (0.759),1.16 (0.731),1.10 (0.686),0.98 (0.446),1.04 (0.605),1.09 (0.682)
E6.9: xgb_vix_only vs hist_rw (DM),1.02 (0.532),1.08 (0.624),0.80 (0.058),0.75 (0.001),0.82 (0.059),0.88 (0.144)
E6.10: xgb_full_macro vs hist_rw (DM),1.09 (0.645),1.10 (0.658),0.86 (0.163),0.92 (0.334),0.91 (0.275),0.97 (0.415)
